# AIC26 — Zero-Shot Object Detection pipeline (Kaggle, T4×2)

End-to-end theo `OD_Pipeline.md`. **Mặc định: CHỈ YOLOE-26-L** (open-vocab, nhanh nhất) → attributes (7×7 grid, 16-color, count) + **DINOv2 crop-embedding**, multi-GPU + checkpoint/resume. Có thể bật ensemble (thêm Grounding DINO + WBF) qua `USE_GDINO=True`; ô §5 để so sánh chất lượng.

**Trước khi chạy — Settings (panel phải):**
1. **Accelerator = GPU T4 ×2**
2. **Internet = ON** (tải weight từ HuggingFace/Ultralytics)
3. Mount cả 2 dataset keyframe (S1, S2) vào `/kaggle/input/`

**Cách chạy:** Run All. Lần đầu nên đặt `MAX_IMAGES=500` (ô Config) để smoke-test ~vài phút, kiểm bbox ở ô §Eyeball, rồi đổi `MAX_IMAGES=None` chạy full.

> Grounding DINO dùng checkpoint công khai `IDEA-Research/grounding-dino-base`. Khi MM-Grounding-DINO Swin-L có sẵn trên HF, chỉ cần đổi `GDINO_MODEL` trong Config.

In [ ]:
# === 1. Cài đặt ===
# Cài sẵn CLIP + ftfy (text-encoder của YOLOE open-vocab) -> tránh YOLOE AutoUpdate giữa chừng
# và cảnh báo "Restart runtime or rerun command". KHÔNG phải lỗi, nhưng cài trước cho sạch.
!pip -q install -U ultralytics "transformers>=4.44" ensemble-boxes timm ftfy 2>/dev/null
!pip -q install -U "git+https://github.com/ultralytics/CLIP.git" 2>/dev/null
import torch, os
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "| GPUs", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print("  -", torch.cuda.get_device_name(i))

In [ ]:
# === 2. Config — chỉnh ở đây ===
import json, os
os.makedirs("/kaggle/working/od_out", exist_ok=True)

MAX_IMAGES = 500   # None = chạy full. Để 500 cho lần test đầu.
USE_GDINO  = False  # MẶC ĐỊNH: CHỈ YOLOE-26-L (nhanh nhất). True = ensemble YOLOE+GDINO. Ô A/B (§5) để so sánh nếu muốn.

# --- Vocab theo THỂ LOẠI (genre-conditional). Mỗi keyframe chỉ prompt concept hợp chương trình. ---
# Áp cho MỌI keyframe:
BASE_VOCAB = ["person","crowd","man","woman","child","text on screen","tv logo",
              "microphone","sign","table","car","motorbike"]

GENRE_VOCAB = {
  "news":     ["reporter","news anchor","official","police officer","soldier","firefighter",
               "ambulance","fire truck","fire","smoke","flood","traffic accident","protest",
               "flag","banner","podium","money","document","weapon","boat","building",
               "market","hospital","factory","farm field","truck"],
  "cycling":  ["bicycle","cyclist","helmet","racing jersey","peloton","road","race number",
               "finish line","spectators","mountain","podium","medal","water bottle","motorbike escort"],
  "lion_dance":["lion dance costume","dragon dance","drum","cymbals","performer","traditional costume",
               "firecrackers","stage","temple","festival decoration","red lantern","flag"],
  "exam":     ["student","teacher","classroom","whiteboard","blackboard","book","notebook","desk",
               "pen","exam paper","school uniform","projector screen","computer"],
  "cooking":  ["chef","food","dish","plate","bowl","pan","pot","stove","cutting board","knife",
               "vegetable","meat","fish","fruit","spice","ingredient","kitchen","cooking oil","rice","noodle"],
  "culture":  ["traditional costume","ao dai","temple","pagoda","statue","handicraft",
               "traditional musical instrument","village","market","festival","lantern","boat",
               "rice field","landscape","river","mountain","old house"],
  "mekong":   ["river","boat","sampan","floating market","rice field","coconut tree","fishing net",
               "fruit","traditional house","water buffalo","bridge","farmer","fish","palm tree",
               "wetland","sunset","riverbank","market"],
  "positive": ["young people","volunteer","student","outdoor activity","sport","group of people",
               "banner","stage","city street","nature","smile"],
}

# Batch -> genre (theo ghi chép của bạn)
GENRE_BY_BATCH = {"L21":"news","L22":"news","L23":"cycling","L24":"lion_dance","L25":"exam",
                  "L26":"cooking","L27":"culture","L28":"mekong","L29":"mekong","L30":"positive"}
for _k in range(1, 21):
    GENRE_BY_BATCH["K%02d" % _k] = "news"   # K01..K20 đều là Thời sự 60 giây (HTV7/HTV9)

PALETTE = {
  "black":[0,0,0],"white":[255,255,255],"gray":[128,128,128],"red":[220,20,20],
  "orange":[255,140,0],"yellow":[240,220,30],"green":[30,160,60],"blue":[30,80,220],
  "navy":[20,30,90],"purple":[130,40,160],"pink":[240,130,180],"brown":[120,70,30],
  "beige":[225,200,160],"cyan":[40,200,210],"maroon":[110,20,30],"olive":[110,110,30],
}

cfg = {
  "input_globs": ["/kaggle/input/**/*.jpg","/kaggle/input/**/*.jpeg","/kaggle/input/**/*.png"],
  "input_root": "/kaggle/input",
  "out_dir": "/kaggle/working/od_out",
  "rel_root": "",                       # "" -> lưu absolute path (dễ mở lại)
  "yoloe_weights": ["yoloe-26l-seg.pt","yoloe-11l-seg.pt","yoloe-v8l-seg.pt"],  # thử lần lượt
  "gdino_model": "IDEA-Research/grounding-dino-base",
  "dino_model": "facebook/dinov2-base",
  "conf_yoloe": 0.25,
  "gdino_box_thr": 0.25,
  "gdino_text_thr": 0.20,
  "wbf_iou": 0.55,
  "wbf_skip": 0.0,
  "use_gdino": USE_GDINO,
  "model_weights": {"yoloe":0.8, "gdino":1.0},
  "vocab_chunk": 20,                    # số phrase / 1 pass GDINO (tránh tràn 256 token)
  "embed_crops": True,
  "crop_conf_min": 0.5,
  "ckpt_every": 200,                    # flush JSONL mỗi N ảnh
  "emb_flush": 20000,                   # flush npy embedding mỗi N crop
  "max_images": MAX_IMAGES,
  "base_vocab": BASE_VOCAB,
  "genre_vocab": GENRE_VOCAB,
  "genre_by_batch": GENRE_BY_BATCH,
  "default_genre": "news",
  "palette": PALETTE,
}
with open("/kaggle/working/od_config.json","w") as f:
    json.dump(cfg, f, ensure_ascii=False, indent=2)
print("Config saved. MAX_IMAGES =", MAX_IMAGES)

In [ ]:
%%writefile od_pipeline_lib.py
# Core OD pipeline library (imported by smoke-test cell AND by od_worker.py)
import os, json, math
from collections import Counter
import numpy as np
from PIL import Image
import torch

def load_cfg(path="/kaggle/working/od_config.json"):
    with open(path) as f:
        return json.load(f)

# ---------- discovery (cache + fast sample) ----------
def discover_images(cfg, use_cache=True):
    # Glob TOÀN BỘ 1 lần rồi cache ra _imglist.txt; lần sau đọc cache (gần như tức thì).
    import glob
    cache = os.path.join(cfg["out_dir"], "_imglist.txt")
    if use_cache and os.path.exists(cache):
        with open(cache) as f:
            imgs = [ln.strip() for ln in f if ln.strip()]
        if imgs:
            return imgs
    imgs = []
    for g in cfg["input_globs"]:
        imgs += glob.glob(g, recursive=True)
    imgs = sorted(set(imgs))
    os.makedirs(cfg["out_dir"], exist_ok=True)
    with open(cache, "w") as f:
        f.write("\n".join(imgs))
    return imgs

def _first_images(root, n):
    found = []
    for dp, dns, fns in os.walk(root):
        for fn in fns:
            if fn.lower().endswith((".jpg", ".jpeg", ".png")):
                found.append(os.path.join(dp, fn))
                if len(found) >= n:
                    return found
    return found

def list_video_dirs(cfg):
    # Liệt kê thư mục video ở ĐỘ SÂU CỐ ĐỊNH bằng glob (KHÔNG đệ quy toàn bộ) -> nhanh.
    import glob
    root = cfg.get("input_root", "/kaggle/input")
    pats = [root + "/*/Keyframes_*/keyframes/*",
            root + "/*/Keyframes_*/*",
            root + "/*/*/keyframes/*",
            root + "/*/keyframes/*",
            root + "/Keyframes_*/keyframes/*"]
    vd = []
    for p in pats:
        vd += [d for d in glob.glob(p) if os.path.isdir(d)]
    vd = sorted(set(vd))
    vv = [d for d in vd if "_V" in os.path.basename(d)]
    return vv or vd

def _images_in(d):
    # Liệt kê ảnh trong 1 thư mục video (scandir, KHÔNG đệ quy).
    out = []
    try:
        for e in os.scandir(d):
            if e.is_file() and e.name.lower().endswith((".jpg", ".jpeg", ".png")):
                out.append(e.path)
    except Exception:
        pass
    return sorted(out)

def quick_sample(cfg, k=6, seed=0):
    # k ảnh ĐA-GENRE: 1 ảnh / mỗi batch khác nhau.
    import random
    from collections import defaultdict
    vdirs = list_video_dirs(cfg)
    if not vdirs:
        root = cfg.get("input_root", "/kaggle/input")
        return _first_images(root, k) if os.path.isdir(root) else []
    bybatch = defaultdict(list)
    for d in vdirs:
        bybatch[batch_prefix(os.path.basename(d))].append(d)
    batches = sorted(bybatch); random.Random(seed).shuffle(batches)
    out = []
    for b in batches:
        im = _images_in(bybatch[b][0])
        if im:
            out.append(im[0])
        if len(out) >= k:
            break
    i = 0
    while len(out) < k and i < len(vdirs):
        im = _images_in(vdirs[i])
        if im and im[0] not in out:
            out.append(im[0])
        i += 1
    return out[:k]

def sample_images(cfg, n=300, seed=0, per_dir=3):
    # ~n ảnh bằng cách DUYỆT THẲNG video-dir (scandir từng thư mục), KHÔNG đệ quy toàn dataset.
    import random
    vdirs = list_video_dirs(cfg)
    if not vdirs:
        return discover_images(cfg)[:n]
    random.Random(seed).shuffle(vdirs)
    out = []
    for d in vdirs:
        imgs = _images_in(d)
        if not imgs:
            continue
        if len(imgs) <= per_dir:
            pick = imgs
        else:
            step = max(1, len(imgs) // per_dir)
            pick = [imgs[j * step] for j in range(per_dir)]
        out += pick
        if len(out) >= n:
            break
    return out[:n]

# ---------- attributes ----------
_PAL = None
def _palette(cfg):
    global _PAL
    if _PAL is None:
        _PAL = {k: np.array(v, dtype=np.float32) for k, v in cfg["palette"].items()}
    return _PAL

def dominant_color(pil, box, cfg):
    x1, y1, x2, y2 = [int(v) for v in box]
    x1 = max(0, x1); y1 = max(0, y1); x2 = max(x1 + 1, x2); y2 = max(y1 + 1, y2)
    crop = pil.crop((x1, y1, x2, y2)).resize((24, 24))
    arr = np.asarray(crop, dtype=np.float32).reshape(-1, 3)
    med = np.median(arr, axis=0)
    best, bd = None, 1e18
    for name, rgb in _palette(cfg).items():
        d = float(((med - rgb) ** 2).sum())
        if d < bd:
            bd, best = d, name
    return best

def grid_cell(box, W, H, n=7):
    cx = (box[0] + box[2]) / 2.0; cy = (box[1] + box[3]) / 2.0
    col = min(n - 1, max(0, int(cx / max(1, W) * n)))
    row = min(n - 1, max(0, int(cy / max(1, H) * n)))
    return [row, col]

def _iou(a, b):
    ix1 = max(a[0], b[0]); iy1 = max(a[1], b[1])
    ix2 = min(a[2], b[2]); iy2 = min(a[3], b[3])
    iw = max(0.0, ix2 - ix1); ih = max(0.0, iy2 - iy1); inter = iw * ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0

# ---------- label space (genre-conditional) ----------
import re as _re
def batch_prefix(video_id):
    m = _re.match(r"([A-Za-z]+\d+)", video_id or "")
    return m.group(1).upper() if m else (video_id or "").upper()

def build_labels(cfg):
    # global union: base + tất cả genre (cho l2i / WBF id-space)
    labels, group = [], {}
    for it in cfg.get("base_vocab", []):
        if it not in group:
            labels.append(it); group[it] = "base"
    for g, items in cfg.get("genre_vocab", {}).items():
        for it in items:
            if it not in group:
                labels.append(it); group[it] = g
    l2i = {l: i for i, l in enumerate(labels)}
    return labels, group, l2i

def allowed_for(video_id, cfg, m):
    # subset concept hợp với thể loại chương trình của keyframe này
    genre = cfg.get("genre_by_batch", {}).get(batch_prefix(video_id), cfg.get("default_genre", "news"))
    seen, out = set(), []
    for x in list(cfg.get("base_vocab", [])) + list(cfg.get("genre_vocab", {}).get(genre, [])):
        if x not in seen and x in m["l2i"]:
            seen.add(x); out.append(x)
    return genre, out

# ---------- model loading ----------
def load_models(device, cfg):
    m = {"device": device}
    labels, group, l2i = build_labels(cfg)
    m["labels"], m["group"], m["l2i"] = labels, group, l2i

    # YOLOE-26-L (open-vocab); fallback các weight cũ nếu chưa có
    from ultralytics import YOLOE
    ym, yname = None, None
    for w in cfg["yoloe_weights"]:
        try:
            ym = YOLOE(w); yname = w; break
        except Exception as e:
            print("[yoloe] load fail", w, "->", repr(e)[:120])
    if ym is None:
        raise RuntimeError("Khong load duoc YOLOE weight nao trong " + str(cfg["yoloe_weights"]))
    try:
        ym.set_classes(labels, ym.get_text_pe(labels))   # text-prompt mode -> label space khop GDINO
    except Exception as e:
        print("[yoloe] set_classes fail:", repr(e)[:120])
    m["yoloe"], m["yoloe_name"] = ym, yname
    print("[yoloe] using", yname)

    # Grounding DINO — CHỈ load khi bật ensemble
    if cfg.get("use_gdino", True):
        from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
        m["gdino_proc"] = AutoProcessor.from_pretrained(cfg["gdino_model"])
        gd = AutoModelForZeroShotObjectDetection.from_pretrained(cfg["gdino_model"]).to(device).eval()
        try:
            gd = gd.half()
        except Exception:
            pass
        m["gdino"] = gd
        print("[gdino] using", cfg["gdino_model"], "dtype", next(gd.parameters()).dtype)
    else:
        print("[gdino] BỎ QUA (chế độ YOLOE-only)")

    # DINOv2 cho crop-embedding (instance retrieval)
    if cfg.get("embed_crops"):
        from transformers import AutoImageProcessor, AutoModel
        m["dino_proc"] = AutoImageProcessor.from_pretrained(cfg["dino_model"])
        dm = AutoModel.from_pretrained(cfg["dino_model"]).to(device).eval()
        try:
            dm = dm.half()
        except Exception:
            pass
        m["dino"] = dm
        print("[dino] using", cfg["dino_model"])
    return m

# ---------- detectors ----------
def _match_label(text, sub):
    t = text.strip().lower().strip(" .")
    if not t:
        return None
    if t in sub:
        return t
    cand = [s for s in sub if s in t or t in s]
    if cand:
        return min(cand, key=len)
    tk = set(t.split()); best, bo = None, 0
    for s in sub:
        o = len(tk & set(s.split()))
        if o > bo:
            bo, best = o, s
    return best if bo > 0 else None

@torch.no_grad()
def detect_yoloe(m, pil, cfg, allowed=None):
    aset = set(allowed) if allowed is not None else None
    boxes, scores, labs = [], [], []
    try:
        # half=False BẮT BUỘC: text-prompt embeddings (set_classes) là fp32,
        # nếu half=True sẽ lỗi "mat1 and mat2 dtype Half != float". YOLOE-26-L nhỏ, fp32 vẫn nhanh.
        r = m["yoloe"].predict(pil, conf=cfg["conf_yoloe"], verbose=False,
                               half=False, device=m["device"])[0]
    except Exception as e:
        print("[yoloe] predict fail:", repr(e)[:120]); return boxes, scores, labs
    if r.boxes is not None and len(r.boxes) > 0:
        xyxy = r.boxes.xyxy.cpu().numpy()
        conf = r.boxes.conf.cpu().numpy()
        cls = r.boxes.cls.cpu().numpy().astype(int)
        names = r.names
        for b, c, k in zip(xyxy, conf, cls):
            nm = names.get(int(k), str(k)) if isinstance(names, dict) else names[int(k)]
            if aset is not None and nm not in aset:
                continue
            boxes.append([float(x) for x in b]); scores.append(float(c)); labs.append(nm)
    return boxes, scores, labs

def _gd_post(proc, out, input_ids, box_thr, text_thr, size):
    try:
        return proc.post_process_grounded_object_detection(
            out, input_ids, threshold=box_thr, text_threshold=text_thr, target_sizes=[size])[0]
    except TypeError:
        return proc.post_process_grounded_object_detection(
            out, input_ids, box_threshold=box_thr, text_threshold=text_thr, target_sizes=[size])[0]

@torch.no_grad()
def detect_gdino(m, pil, cfg, allowed=None):
    proc, gd, dev = m["gdino_proc"], m["gdino"], m["device"]
    labels = list(allowed) if allowed is not None else m["labels"]
    chunk = cfg["vocab_chunk"]
    W, H = pil.size
    boxes, scores, labs = [], [], []
    for i in range(0, len(labels), chunk):
        sub = labels[i:i + chunk]
        text = (". ".join(sub) + " .").lower()
        inp = proc(images=pil, text=text, return_tensors="pt").to(dev)
        if next(gd.parameters()).dtype == torch.float16 and "pixel_values" in inp:
            inp["pixel_values"] = inp["pixel_values"].half()
        out = gd(**inp)
        res = _gd_post(proc, out, inp["input_ids"], cfg["gdino_box_thr"], cfg["gdino_text_thr"], (H, W))
        bb = res["boxes"].cpu().numpy(); ss = res["scores"].cpu().numpy(); tt = res["labels"]
        for b, s, t in zip(bb, ss, tt):
            lab = _match_label(t, sub)
            if lab is None:
                continue
            boxes.append([float(x) for x in b]); scores.append(float(s)); labs.append(lab)
    return boxes, scores, labs

# ---------- fusion ----------
def wbf_merge(yb, ys, yl, gb, gs, gl, W, H, m, cfg):
    from ensemble_boxes import weighted_boxes_fusion
    l2i = m["l2i"]; i2l = {v: k for k, v in l2i.items()}

    def pack(bl, sl, ll):
        nb, ns, ni = [], [], []
        for b, s, l in zip(bl, sl, ll):
            if l not in l2i:
                continue
            nb.append([min(1, max(0, b[0]/W)), min(1, max(0, b[1]/H)),
                       min(1, max(0, b[2]/W)), min(1, max(0, b[3]/H))])
            ns.append(s); ni.append(l2i[l])
        return nb, ns, ni

    boxes_list, scores_list, labels_list, weights = [], [], [], []
    yn = pack(yb, ys, yl); gn = pack(gb, gs, gl)
    if yn[0]:
        boxes_list.append(yn[0]); scores_list.append(yn[1]); labels_list.append(yn[2]); weights.append(cfg["model_weights"]["yoloe"])
    if gn[0]:
        boxes_list.append(gn[0]); scores_list.append(gn[1]); labels_list.append(gn[2]); weights.append(cfg["model_weights"]["gdino"])
    if not boxes_list:
        return [], [], [], []
    fb, fs, fl = weighted_boxes_fusion(boxes_list, scores_list, labels_list,
                                       weights=weights, iou_thr=cfg["wbf_iou"], skip_box_thr=cfg["wbf_skip"])
    out_boxes = [[b[0]*W, b[1]*H, b[2]*W, b[3]*H] for b in fb]
    out_labels = [i2l.get(int(round(x)), "unknown") for x in fl]
    out_scores = [float(s) for s in fs]
    # source attribution (model nao gop vao box)
    orig = {"yoloe": (yb, yl), "gdino": (gb, gl)}
    src = []
    for ob, ol in zip(out_boxes, out_labels):
        s = []
        for name, (bl, ll) in orig.items():
            for bb, lll in zip(bl, ll):
                if lll == ol and _iou(ob, bb) >= cfg["wbf_iou"]:
                    s.append(name); break
        src.append(s or ["fused"])
    return out_boxes, out_scores, out_labels, src

# ---------- per-image ----------
@torch.no_grad()
def process_image(m, path, cfg, rel_root=""):
    pil = Image.open(path).convert("RGB"); W, H = pil.size
    vid = os.path.basename(os.path.dirname(path))
    frame = os.path.splitext(os.path.basename(path))[0]
    genre, allowed = allowed_for(vid, cfg, m)
    yb, ys, yl = detect_yoloe(m, pil, cfg, allowed)
    if cfg.get("use_gdino", True):
        gb, gs, gl = detect_gdino(m, pil, cfg, allowed)
        fb, fs, fl, fsrc = wbf_merge(yb, ys, yl, gb, gs, gl, W, H, m, cfg)
    else:
        # YOLOE-only: dùng box trực tiếp (YOLOE đã NMS), KHÔNG qua WBF
        fb, fs, fl, fsrc = yb, ys, yl, [["yoloe"]] * len(yb)
    group = m["group"]
    dets, emb_jobs = [], []
    for box, score, lab, src in zip(fb, fs, fl, fsrc):
        d = {"label": lab, "label_group": group.get(lab, "other"),
             "bbox": [round(float(x), 1) for x in box], "conf": round(float(score), 3),
             "source": src, "grid_cell": grid_cell(box, W, H),
             "color": dominant_color(pil, box, cfg)}
        if cfg.get("embed_crops") and score >= cfg["crop_conf_min"]:
            emb_jobs.append((len(dets), box))
        dets.append(d)
    counts = dict(Counter(d["label"] for d in dets))
    rec = {"keyframe_id": [vid, frame], "video_id": vid, "frame": frame, "genre": genre,
           "path": (os.path.relpath(path, rel_root) if rel_root else path),
           "width": W, "height": H, "detections": dets, "counts": counts}
    return rec, pil, emb_jobs

@torch.no_grad()
def embed_crops(m, pil, emb_jobs, cfg):
    if not emb_jobs:
        return np.zeros((0, 1), dtype=np.float16)
    dp, dm, dev = m["dino_proc"], m["dino"], m["device"]
    crops = [pil.crop(tuple(int(v) for v in box)) for _, box in emb_jobs]
    inp = dp(images=crops, return_tensors="pt").to(dev)
    if next(dm.parameters()).dtype == torch.float16:
        inp["pixel_values"] = inp["pixel_values"].half()
    out = dm(**inp)
    feat = out.last_hidden_state[:, 0]                       # CLS token
    feat = torch.nn.functional.normalize(feat, dim=-1)
    return feat.cpu().float().numpy().astype(np.float16)


In [ ]:
# === 4. Smoke-test NHANH + Eyeball (mẫu đa-genre, KHÔNG quét toàn bộ dataset) ===
import importlib, json, torch, gc, os
import od_pipeline_lib as L
importlib.reload(L)
cfg = L.load_cfg()

sample = L.quick_sample(cfg, k=6)   # nhanh: chỉ stat vài thư mục Keyframes_*, lấy mẫu nhiều genre
assert sample, "Không thấy ảnh — kiểm tra dataset đã mount vào /kaggle/input chưa."
print("Mẫu nhanh (đa genre):")
for p in sample:
    vid = os.path.basename(os.path.dirname(p))
    print("   %-4s | %s / %s" % (L.batch_prefix(vid), vid, os.path.basename(p)))

dev = "cuda:0" if torch.cuda.is_available() else "cpu"
sm = L.load_models(dev, cfg)

import matplotlib.pyplot as plt
import matplotlib.patches as patches
cols = 3; rows = (len(sample) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
axes = axes.ravel() if hasattr(axes, "ravel") else [axes]
last = None
for ax, p in zip(axes, sample):
    rec, pil, _ = L.process_image(sm, p, cfg); last = rec
    ax.imshow(pil); ax.axis("off")
    ax.set_title("[%s] %s/%s · %dobj" % (rec["genre"], rec["video_id"], rec["frame"], len(rec["detections"])), fontsize=8)
    for d in rec["detections"]:
        x1, y1, x2, y2 = d["bbox"]
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="lime", lw=1.5))
        ax.text(x1, max(0, y1-2), "%s %.2f" % (d["label"], d["conf"]), color="black",
                fontsize=6, bbox=dict(facecolor="lime", alpha=0.7, pad=0))
for ax in axes[len(sample):]: ax.axis("off")
plt.tight_layout(); plt.show()
if last: print(json.dumps(last, ensure_ascii=False, indent=2)[:1200])

del sm; gc.collect(); torch.cuda.empty_cache()
print("Đã giải phóng model smoke-test.")

## §5. A/B test — YOLOE-26-L đơn lẻ vs Ensemble (YOLOE + GDINO)

Không có ground-truth nhãn, nên ta đo **trade-off bằng metric tương đối**: (1) tốc độ mỗi ảnh & ngoại suy 400k, (2) GDINO **thêm** được bao nhiêu box/label mà YOLOE miss, (3) các box thêm thuộc class nào (có phải rare/news entity đáng giá không).

**Ô này TÙY CHỌN** (mặc định đã là YOLOE-only). Chỉ chạy nếu bạn muốn cân nhắc lại ensemble: nó load cả 2 model trên ~300 ảnh random → verdict xem GDINO có đáng thêm thời gian không. Nếu đã quyết YOLOE-only thì **bỏ qua ô này**.

In [ ]:
# === 5. A/B: YOLOE-26-L alone vs Ensemble — có đáng đổi thời gian không? ===
import glob, random, time, importlib, os
import numpy as np, torch, gc
import od_pipeline_lib as L
importlib.reload(L)
cfg = L.load_cfg()
cfg_ab = dict(cfg); cfg_ab["use_gdino"] = True   # A/B cần CẢ 2 model để so sánh, dù run chính là YOLOE-only
m = L.load_models("cuda:0" if torch.cuda.is_available() else "cpu", cfg_ab)

sample = L.sample_images(cfg, n=300)   # duyệt THẲNG video-dir (scandir), KHÔNG đệ quy toàn bộ dataset
N = len(sample)
print("A/B trên", N, "ảnh (lấy nhanh, không glob đệ quy)")

from PIL import Image
t_y = t_g = 0.0
per_y, per_e = [], []
labels_y, labels_e = set(), set()
gdino_unique = 0; ens_total = 0; uniq_by_class = {}

for p in sample:
    try:
        pil = Image.open(p).convert("RGB"); W, H = pil.size
    except Exception:
        continue
    vid = os.path.basename(os.path.dirname(p)); _, allowed = L.allowed_for(vid, cfg, m)
    t0 = time.time(); yb, ys, yl = L.detect_yoloe(m, pil, cfg, allowed); t_y += time.time() - t0
    t0 = time.time(); gb, gs, gl = L.detect_gdino(m, pil, cfg, allowed); t_g += time.time() - t0
    fb, fs, fl, fsrc = L.wbf_merge(yb, ys, yl, gb, gs, gl, W, H, m, cfg)
    per_y.append(len(yb)); per_e.append(len(fb))
    labels_y |= set(yl); labels_e |= set(fl)
    for box, lab in zip(fb, fl):
        ens_total += 1
        matched = any((lll == lab and L._iou(box, bb) >= cfg["wbf_iou"]) for bb, lll in zip(yb, yl))
        if not matched:
            gdino_unique += 1
            uniq_by_class[lab] = uniq_by_class.get(lab, 0) + 1

ms_y = t_y / N * 1000; ms_e = (t_y + t_g) / N * 1000
h_y = t_y / N * 400000 / 3600 / 2; h_e = (t_y + t_g) / N * 400000 / 3600 / 2
share = 100 * gdino_unique / max(1, ens_total)

print("================= A/B trên %d ảnh (1×T4) =================" % N)
print("[TỐC ĐỘ]")
print("  YOLOE-only : %6.1f ms/ảnh   -> 400k ~ %.1fh (2×T4)" % (ms_y, h_y))
print("  Ensemble   : %6.1f ms/ảnh   -> 400k ~ %.1fh (2×T4)" % (ms_e, h_e))
print("  Ensemble chậm hơn %.1f×, tốn thêm ~%.1fh cho 400k" % (ms_e / max(1e-9, ms_y), h_e - h_y))
print("[ĐỘ PHỦ]")
print("  Box TB/ảnh : YOLOE %.2f | Ensemble %.2f (+%.0f%%)" % (
      np.mean(per_y), np.mean(per_e), 100*(np.mean(per_e)-np.mean(per_y))/max(1e-9,np.mean(per_y))))
print("  Loại label : YOLOE %d | Ensemble %d" % (len(labels_y), len(labels_e)))
print("  Label chỉ Ensemble có:", sorted(labels_e - labels_y))
print("[ĐÓNG GÓP GDINO]")
print("  Box GDINO thêm (YOLOE miss): %d/%d = %.1f%%" % (gdino_unique, ens_total, share))
print("  Theo class:", sorted(uniq_by_class.items(), key=lambda x: -x[1])[:15])
print("=========================================================")
if share < 8:
    print("VERDICT: GDINO thêm <8%% -> YOLOE-26-L ĐƠN LẺ đáng giá hơn (đặt USE_GDINO=False).")
elif share < 20:
    print("VERDICT: 8-20%% -> tùy: nếu class thêm vào là rare/news quan trọng thì giữ Ensemble.")
else:
    print("VERDICT: >20%% (nhiều rare class) -> ENSEMBLE đáng đổi thời gian (USE_GDINO=True).")
print("LƯU Ý: con số chỉ là proxy. Quyết định cuối dựa vào: các class GDINO thêm ở trên CÓ quan trọng cho query tin tức không.")

# ---- minh hoạ: đỏ = box ensemble mà YOLOE miss ----
import matplotlib.pyplot as plt, matplotlib.patches as patches
vis = sample[:6]; cols = 3; rows = (len(vis)+cols-1)//cols
fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 5*rows)); axes = axes.ravel()
for ax, p in zip(axes, vis):
    try: pil = Image.open(p).convert("RGB"); W, H = pil.size
    except Exception: ax.axis("off"); continue
    vid = os.path.basename(os.path.dirname(p)); _, allowed = L.allowed_for(vid, cfg, m)
    yb, ys, yl = L.detect_yoloe(m, pil, cfg, allowed); gb, gs, gl = L.detect_gdino(m, pil, cfg, allowed)
    fb, fs, fl, fsrc = L.wbf_merge(yb, ys, yl, gb, gs, gl, W, H, m, cfg)
    ax.imshow(pil); ax.axis("off"); add = 0
    for b in yb:
        ax.add_patch(patches.Rectangle((b[0],b[1]), b[2]-b[0], b[3]-b[1], fill=False, edgecolor="yellow", lw=1.1))
    for box, lab in zip(fb, fl):
        if not any((lll == lab and L._iou(box, bb) >= cfg["wbf_iou"]) for bb, lll in zip(yb, yl)):
            add += 1
            ax.add_patch(patches.Rectangle((box[0],box[1]), box[2]-box[0], box[3]-box[1], fill=False, edgecolor="red", lw=1.8))
            ax.text(box[0], max(0,box[1]-2), lab, color="white", fontsize=6, bbox=dict(facecolor="red", alpha=0.8, pad=0))
    ax.set_title("YOLOE %d (vàng) + GDINO thêm %d (đỏ)" % (len(yb), add), fontsize=8)
for ax in axes[len(vis):]: ax.axis("off")
plt.tight_layout(); plt.show()

del m; gc.collect(); torch.cuda.empty_cache()

In [ ]:
%%writefile od_worker.py
# Worker chạy 1 shard trên 1 GPU (CUDA_VISIBLE_DEVICES set từ ngoài). Có checkpoint/resume.
import argparse, os, json, glob, time
import numpy as np

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--rank", type=int, default=0)
    ap.add_argument("--world-size", type=int, default=1)
    ap.add_argument("--config", default="/kaggle/working/od_config.json")
    a = ap.parse_args()

    import torch
    import od_pipeline_lib as L
    cfg = L.load_cfg(a.config)
    dev = "cuda:0" if torch.cuda.is_available() else "cpu"   # mỗi process chỉ thấy 1 GPU

    imgs = L.discover_images(cfg)          # đọc cache _imglist.txt (launcher đã build trước)
    if cfg.get("max_images"):
        imgs = imgs[:cfg["max_images"]]
    shard = [p for i, p in enumerate(imgs) if i % a.world_size == a.rank]

    out_dir = cfg["out_dir"]; os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, "dets_rank%d.jsonl" % a.rank)

    done = set()
    if os.path.exists(out_path):
        with open(out_path) as f:
            for line in f:
                try:
                    done.add(tuple(json.loads(line)["keyframe_id"]))
                except Exception:
                    pass
    print("[rank%d] shard=%d done=%d dev=%s" % (a.rank, len(shard), len(done), dev), flush=True)

    models = L.load_models(dev, cfg)
    fout = open(out_path, "a")

    emb_buf, emb_ids = [], []
    emb_part = len(glob.glob(os.path.join(out_dir, "crops_rank%d_part*.npy" % a.rank)))
    rel_root = cfg.get("rel_root", "")
    t0 = time.time(); n = 0

    def flush_emb():
        nonlocal emb_buf, emb_ids, emb_part
        if not emb_buf:
            return
        np.save(os.path.join(out_dir, "crops_rank%d_part%d.npy" % (a.rank, emb_part)), np.stack(emb_buf))
        with open(os.path.join(out_dir, "crops_rank%d_part%d.json" % (a.rank, emb_part)), "w") as g:
            json.dump(emb_ids, g)
        emb_buf, emb_ids = [], []; emb_part += 1

    for p in shard:
        vid = os.path.basename(os.path.dirname(p))
        frame = os.path.splitext(os.path.basename(p))[0]
        if (vid, frame) in done:
            continue
        try:
            rec, pil, emb_jobs = L.process_image(models, p, cfg, rel_root)
            if cfg.get("embed_crops") and emb_jobs:
                feats = L.embed_crops(models, pil, emb_jobs, cfg)
                for (di, _), vec in zip(emb_jobs, feats):
                    cid = "%d:%d:%d" % (a.rank, emb_part, len(emb_ids))
                    rec["detections"][di]["crop_emb_id"] = cid
                    emb_buf.append(vec); emb_ids.append(cid)
            fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
            n += 1
        except Exception as e:
            print("[rank%d] ERR %s: %s" % (a.rank, p, repr(e)[:160]), flush=True)

        if n > 0 and n % cfg["ckpt_every"] == 0:
            fout.flush(); os.fsync(fout.fileno())
            sp = time.time() - t0; rate = n / max(1e-9, sp)
            eta = (len(shard) - len(done) - n) / max(1e-9, rate) / 3600.0
            print("[rank%d] %d new | %.2f img/s | ETA %.2fh" % (a.rank, n, rate, eta), flush=True)
        if cfg.get("embed_crops") and len(emb_buf) >= cfg.get("emb_flush", 20000):
            flush_emb()

    fout.flush(); fout.close()
    if cfg.get("embed_crops"):
        flush_emb()
    print("[rank%d] FINISHED %d new records" % (a.rank, n), flush=True)

if __name__ == "__main__":
    main()


In [ ]:
# === 6. Chạy full đa-GPU (mỗi T4 một process, song song) ===
import subprocess, os, threading, torch
import od_pipeline_lib as L

_all = L.discover_images(L.load_cfg())   # glob TOÀN BỘ 1 lần -> cache; workers chỉ đọc cache
print("Tổng ảnh (đã cache):", len(_all))

ngpu = torch.cuda.device_count()
world = max(1, ngpu)
print("Khởi chạy", world, "worker (1/GPU)…")

procs = []
for r in range(world):
    env = dict(os.environ)
    if ngpu > 0:
        env["CUDA_VISIBLE_DEVICES"] = str(r)
    p = subprocess.Popen(
        ["python", "od_worker.py", "--rank", str(r), "--world-size", str(world)],
        env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    procs.append(p)

def pump(p):
    for line in p.stdout:
        print(line, end="")

threads = [threading.Thread(target=pump, args=(p,), daemon=True) for p in procs]
for t in threads: t.start()
for p in procs: p.wait()
for t in threads: t.join()
print("=== Tất cả worker đã xong ===")# === 6. Chạy full đa-GPU (mỗi T4 một process, song song) ===
import subprocess, os, threading, torch
import od_pipeline_lib as L

_all = L.discover_images(L.load_cfg())   # glob TOÀN BỘ 1 lần -> cache; workers chỉ đọc cache
print("Tổng ảnh (đã cache):", len(_all))

ngpu = torch.cuda.device_count()
world = max(1, ngpu)
print("Khởi chạy", world, "worker (1/GPU)…")

procs = []
for r in range(world):
    env = dict(os.environ)
    if ngpu > 0:
        env["CUDA_VISIBLE_DEVICES"] = str(r)
    p = subprocess.Popen(
        ["python", "od_worker.py", "--rank", str(r), "--world-size", str(world)],
        env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    procs.append(p)

def pump(p):
    for line in p.stdout:
        print(line, end="")

threads = [threading.Thread(target=pump, args=(p,), daemon=True) for p in procs]
for t in threads: t.start()
for p in procs: p.wait()
for t in threads: t.join()
print("=== Tất cả worker đã xong ===")

In [ ]:
# === 7. Gộp kết quả + thống kê ===
import glob, json, os
from collections import Counter
cfg = json.load(open("/kaggle/working/od_config.json"))
out = cfg["out_dir"]
final = os.path.join(out, "detections.jsonl")

n = 0
with open(final, "w") as w:
    for f in sorted(glob.glob(os.path.join(out, "dets_rank*.jsonl"))):
        for line in open(f):
            w.write(line); n += 1
print("Gộp", n, "keyframe ->", final)

lab = Counter(); col = Counter(); ndet = 0; only = Counter(); gen = Counter()
for line in open(final):
    r = json.loads(line)
    gen[r.get("genre", "?")] += 1
    for d in r["detections"]:
        lab[d["label"]] += 1; col[d["color"]] += 1; ndet += 1
        if len(d["source"]) == 1: only[d["source"][0]] += 1
print("Tổng detection:", ndet, "| trung bình/keyframe:", round(ndet/max(1,n), 2))
print("Top 20 label:", lab.most_common(20))
print("Màu:", col.most_common(8))
print("Box chỉ 1 model thấy (proxy độ tin):", dict(only))
print("Keyframe theo genre:", gen.most_common())

In [ ]:
# === 8. Validation §7 — eyeball ngẫu nhiên (kiểm vocab/threshold) ===
import json, random
import matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image
recs = [json.loads(l) for l in open(final)]
random.seed(1)
sample = random.sample(recs, min(8, len(recs)))
cols = 4; rows = (len(sample)+cols-1)//cols
fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
axes = axes.ravel() if hasattr(axes, "ravel") else [axes]
COLOR = {"yoloe":"yellow","gdino":"cyan","fused":"lime"}
for ax, r in zip(axes, sample):
    try:
        pil = Image.open(r["path"]).convert("RGB")
    except Exception as e:
        ax.set_title("open fail"); ax.axis("off"); continue
    ax.imshow(pil); ax.axis("off")
    ax.set_title(f"{r['video_id']}/{r['frame']} · {len(r['detections'])}obj", fontsize=8)
    for d in r["detections"]:
        x1,y1,x2,y2 = d["bbox"]
        c = COLOR.get(d["source"][0], "lime")
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1, fill=False, edgecolor=c, lw=1.4))
        ax.text(x1, max(0,y1-2), f"{d['label']} {d['conf']:.2f}", color="black",
                fontsize=6, bbox=dict(facecolor=c, alpha=0.7, pad=0))
for ax in axes[len(sample):]: ax.axis("off")
plt.tight_layout(); plt.show()
print("Vàng=YOLOE  Cyan=GDINO  Lime=cả hai (high-confidence).")
print(">> Nếu vocab miss concept tin tức -> sửa NEWS_VOCAB ở ô Config rồi chạy lại.")

## Kết quả & bước tiếp theo

**Output trong `/kaggle/working/od_out/`:**
- `detections.jsonl` — mỗi dòng 1 keyframe theo schema §6 của `OD_Pipeline.md` (`label, bbox, conf, source, color, grid_cell, crop_emb_id, counts`).
- `crops_rank*_part*.npy` + `.json` — DINOv2 crop-embeddings (id ↔ vector) cho instance retrieval.

**Tải về / dùng tiếp:** `detections.jsonl` nạp thẳng vào Postgres JSONB (faceted/spatial/color filter); crop-embeddings nạp vào Milvus.

**Chế độ chạy:** mặc định `USE_GDINO=False` → chỉ YOLOE-26-L (nhanh nhất, ~1-2h cho 400k trên 2×T4). Muốn thêm recall rare-class thì đặt `USE_GDINO=True` (bật ensemble + WBF). Ô §5 giúp so sánh trước khi quyết.

**Đổi sang chạy FULL:** ở ô Config đặt `MAX_IMAGES=None`, Run All. Notebook tự **resume** (bỏ qua keyframe đã có trong JSONL) nên cứ chạy tiếp qua nhiều session 12h.

**Nâng độ chính xác (nếu có budget >60h):** đổi `GDINO_MODEL` sang checkpoint MM-Grounding-DINO Swin-L khi có trên HF; bật pass Qwen3-VL cho subset đông người (đếm) — thêm ở giai đoạn online, không cần ở đây.

**Lưu ý vận hành:**
- Nếu lần đầu OOM ở ô §6: Restart kernel, chạy lại **bỏ qua ô §4 smoke-test** (để dành VRAM cho 2 worker).
- I/O đọc ảnh nhỏ là nghẽn ẩn — nếu chậm, cân nhắc đóng gói keyframe thành tar/webdataset.